In [44]:
import os
os.getcwd()

'/Users/komalpreet/Desktop/Github/Amazon_Product_Query_Assistant'

In [45]:
os.chdir('/Users/komalpreet/Desktop/Github/Amazon_Product_Query_Assistant/')

In [4]:
from datasets import load_dataset

reviews = load_dataset(
    "McAuley-Lab/Amazon-Reviews-2023",
    "raw_review_All_Beauty",
    split="full",
    trust_remote_code=True,
)

meta = load_dataset(
    "McAuley-Lab/Amazon-Reviews-2023",
    "raw_meta_All_Beauty",
    split="full",
    trust_remote_code=True,
)

print(reviews[0])
print(meta[0])

/Users/komalpreet/miniconda3/envs/inflection/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating full split: 701528 examples [00:02, 278314.32 examples/s]
Generating full split: 100%|██████████████████████████████████████████████| 112590/112590 [00:03<00:00, 29851.10 examples/s]

{'rating': 5.0, 'title': 'Such a lovely scent but not overpowering.', 'text': "This spray is really nice. It smells really good, goes on really fine, and does the trick. I will say it feels like you need a lot of it though to get the texture I want. I have a lot of hair, medium thickness. I am comparing to other brands with yucky chemicals so I'm gonna stick with this. Try it!", 'images': [], 'asin': 'B00YQ6X8EO', 'parent_asin': 'B00YQ6X8EO', 'user_id': 'AGKHLEW2SOWHNMFQIJGBECAF7INQ', 'timestamp': 1588687728923, 'helpful_vote': 0, 'verified_purchase': True}
{'main_category': 'All Beauty', 'title': 'Howard LC0008 Leather Conditioner, 8-Ounce (4-Pack)', 'average_rating': 4.8, 'rating_number': 10, 'features': [], 'description': [], 'price': 'None', 'images': {'hi_res': [None, 'https://m.media-amazon.com/images/I/71i77AuI9xL._SL1500_.jpg'], 'large': ['https://m.media-amazon.com/images/I/41qfjSfqNyL.jpg', 'https://m.media-amazon.com/images/I/41w2yznfuZL.jpg'], 'thumb': ['https://m.media-ama

In [5]:
# how many records?
print("Reviews:", len(reviews))
print("Meta:", len(meta))

# check fields
print("\nReview fields:", reviews[0].keys())
print("Meta fields:", meta[0].keys())

Reviews: 701528
Meta: 112590

Review fields: dict_keys(['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase'])
Meta fields: dict_keys(['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together', 'subtitle', 'author'])


In [6]:
import pandas as pd

df_meta = pd.DataFrame(meta)
print(df_meta[['title', 'price', 'average_rating', 'rating_number', 'store', 'features', 'description']].isnull().sum())
print("\nPrice sample:", df_meta['price'].dropna().head())

title                 0
price                 0
average_rating        0
rating_number         0
store             11331
features              0
description           0
dtype: int64

Price sample: 0    None
1    None
2    None
3    None
4    None
Name: price, dtype: str


In [7]:
# check price more carefully
print("Unique price samples:", df_meta['price'].unique()[:10])

# check features and description — are they actually populated?
print("\nEmpty features:", df_meta['features'].apply(lambda x: len(x) == 0).sum())
print("Empty description:", df_meta['description'].apply(lambda x: len(x) == 0).sum())

# check details
print("\nDetails sample:", df_meta['details'].dropna().iloc[0])

Unique price samples: <ArrowStringArray>
[ 'None',  '6.99', '86.95',  '79.5',  '5.99',  '29.8',  '24.0', '22.49',
 '11.99',  '50.0']
Length: 10, dtype: str

Empty features: 95213
Empty description: 93428

Details sample: {"Package Dimensions": "7.1 x 5.5 x 3 inches; 2.38 Pounds", "UPC": "617390882781"}


In [8]:
df_reviews = pd.DataFrame(reviews)

# how many reviews per product?
reviews_per_product = df_reviews.groupby('parent_asin').size()
print(reviews_per_product.describe())

# how many products have at least 1 review?
print("\nProducts with reviews:", reviews_per_product.shape[0])

# text quality
print("\nEmpty review text:", df_reviews['text'].apply(lambda x: len(str(x).strip()) == 0).sum())
print("Verified purchase %:", df_reviews['verified_purchase'].mean() * 100)

count    112565.000000
mean          6.232204
std          25.189840
min           1.000000
25%           1.000000
50%           2.000000
75%           4.000000
max        1962.000000
dtype: float64

Products with reviews: 112565

Empty review text: 720
Verified purchase %: 90.51228176209645


In [11]:
import ast

# how populated are details?
print("Empty details:", df_meta['details'].apply(lambda x: x == '{}' or x == '' or x is None).sum())

# parse and check keys
from collections import Counter
all_keys = Counter()
for d in df_meta['details']:
    try:
        parsed = ast.literal_eval(d)
        if parsed:
            all_keys.update(parsed.keys())
    except:
        continue

print("\nTop 10 detail keys:", all_keys.most_common(10))

Empty details: 4512

Top 10 detail keys: [('Brand', 72113), ('Package Dimensions', 67825), ('UPC', 60252), ('Is Discontinued By Manufacturer', 39527), ('Item Form', 32439), ('Material', 30942), ('Hair Type', 26719), ('Unit Count', 23855), ('Product Dimensions', 23277), ('Age Range (Description)', 22929)]


In [16]:
import json
from pathlib import Path

Path("data/raw").mkdir(parents=True, exist_ok=True)

# save raw reviews
with open("data/raw/reviews_raw.jsonl", "w") as f:
    for row in reviews:
        f.write(json.dumps(dict(row)) + "\n")

# save raw meta
with open("data/raw/meta_raw.jsonl", "w") as f:
    for row in meta:
        f.write(json.dumps(dict(row)) + "\n")

print("Done!")

Done!


In [17]:
import os

reviews_size = os.path.getsize("data/raw/reviews_raw.jsonl")
meta_size    = os.path.getsize("data/raw/meta_raw.jsonl")

print(f"Reviews: {reviews_size / 1024 / 1024:.1f} MB")
print(f"Meta:    {meta_size / 1024 / 1024:.1f} MB")

Reviews: 311.5 MB
Meta:    195.9 MB


In [30]:
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s")

import sys
sys.path.append(".")

from src.preprocessor import build_products

products = build_products(meta, reviews)
print(f"Total products: {len(products)}")
print(f"\nSample search_text:\n{products[0]['search_text']}")

2026-05-02 17:31:12,301 Grouping reviews by parent_asin...
2026-05-02 17:31:23,069   112,565 unique products have reviews
2026-05-02 17:31:23,069 Building product documents...
2026-05-02 17:31:28,705 Total products built: 112,590
2026-05-02 17:31:29,585 Saved → data/processed/products.jsonl


Total products: 112590

Sample search_text:
Howard LC0008 Leather Conditioner, 8-Ounce (4-Pack) | Store: Howard Products | Reviews: Absolutely fabulous - I will never use anything else. This is amazing.  I bought one at a local shop....I then looked at Amazon and purchased 5 more.  My beautiful leather sofa and chairs now look like new.  You will not believe it. Of all leather products I have used this one seems to be the best. I would recommend this product as the last coat for leather (if you have really old dried leather I would recommend covering it with neats foot oil first) the wax in this product will build up some water resistance as a last coat.  But if you want to rejuvenate and protect recent leather I have not found a product any better than this one. Product works as advertised.. I have a 10 year old Bernhart leather chair and ottoman that has been in a room with a fireplace it's entire life. The leather has become very dry as we mostly used spray on leather conditioners. 

In [31]:
import pandas as pd
df_meta = pd.DataFrame(meta)
print(df_meta['price'].value_counts().head(10))
print("\nNull/None count:", df_meta['price'].apply(lambda x: x is None or str(x).strip().lower() in ('none', '', 'null')).sum())

price
None     94886
9.99       631
19.99      363
14.99      314
8.99       283
7.99       280
6.99       275
11.99      252
12.99      252
5.99       243
Name: count, dtype: int64

Null/None count: 94886


In [32]:
# check search_text length
import numpy as np
lengths = [len(p['search_text'].split()) for p in products]
print(f"Avg words: {np.mean(lengths):.0f}")
print(f"Max words: {np.max(lengths)}")
print(f"95th percentile: {np.percentile(lengths, 95):.0f}")

Avg words: 133
Max words: 2807
95th percentile: 387


In [33]:
print(products[0]['top_reviews'])

[{'title': 'Absolutely fabulous - I will never use anything else', 'text': 'This is amazing.  I bought one at a local shop....I then looked at Amazon and purchased 5 more.  My beautiful leather sofa and chairs now look like new.  You will not believe it.', 'rating': 5.0}, {'title': 'Of all leather products I have used this one seems to be the best', 'text': 'I would recommend this product as the last coat for leather (if you have really old dried leather I would recommend covering it with neats foot oil first) the wax in this product will build up some water resistance as a last coat.  But if you want to rejuvenate and protect recent leather I have not found a product any better than this one.', 'rating': 5.0}, {'title': 'Product works as advertised.', 'text': "I have a 10 year old Bernhart leather chair and ottoman that has been in a room with a fireplace it's entire life. The leather has become very dry as we mostly used spray on leather conditioners. I bought 4 bottles of this cream

In [34]:
import importlib
import sys

# remove cached module
if 'src.preprocessor' in sys.modules:
    del sys.modules['src.preprocessor']

from src.preprocessor import build_products
products = build_products(meta, reviews)

import numpy as np
lengths = [len(p['search_text'].split()) for p in products]
print(f"Avg words: {np.mean(lengths):.0f}")
print(f"Max words: {np.max(lengths)}")
print(f"95th percentile: {np.percentile(lengths, 95):.0f}")

2026-05-02 17:31:55,788 Grouping reviews by parent_asin...
2026-05-02 17:32:07,093   112,565 unique products have reviews
2026-05-02 17:32:07,094 Building product documents...
2026-05-02 17:32:12,707 Total products built: 112,590
2026-05-02 17:32:13,520 Saved → data/processed/products.jsonl


Avg words: 108
Max words: 2305
95th percentile: 299


In [35]:
import sys
sys.path.append(".")

import nltk
nltk.download("stopwords")

from src.utils import tokenize, build_corpus

# test tokenizer
print(tokenize("Best moisturizer for sensitive skin!"))

# build corpus
corpus, tokenized_corpus = build_corpus(products)
print(f"Corpus size: {len(corpus)}")
print(f"Sample tokens: {tokenized_corpus[0][:10]}")

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/komalpreet/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


ImportError: cannot import name 'tokenize' from 'src.utils' (/Users/komalpreet/Desktop/Github/Amazon_Product_Query_Assistant/src/utils.py)

In [37]:
import sys

# remove cached module
if 'src.utils' in sys.modules:
    del sys.modules['src.utils']

from src.utils import tokenize, build_corpus

# test tokenizer
print(tokenize("Best moisturizer for sensitive skin!"))

# build corpus
corpus, tokenized_corpus = build_corpus(products)
print(f"Corpus size: {len(corpus)}")
print(f"Sample tokens: {tokenized_corpus[0][:10]}")

2026-05-02 17:39:10,557 Building corpus from 112,590 products...


['best', 'moisturizer', 'sensitive', 'skin']


2026-05-02 17:39:12,485 Corpus built: 112,590 documents


Corpus size: 112590
Sample tokens: ['howard', 'lc0008', 'leather', 'conditioner', '8ounce', '4pack', 'store', 'howard', 'products', 'reviews']


In [38]:
import sys
!{sys.executable} -m pip install rank_bm25

In [40]:
bm25 = build_bm25(tokenized_corpus)

results = search_bm25(bm25, products, "moisturizer for sensitive skin", top_k=5)
for r in results:
    print(f"{r['bm25_score']:.4f} | {r['title']}")

2026-05-03 12:58:50,564 Building BM25 index over 112,590 documents...
2026-05-03 12:58:52,431 BM25 index saved → data/processed/bm25_index.pkl
2026-05-03 12:58:53,240 Tokenized corpus saved → data/processed/tokenized_corpus.pkl


16.8432 | Simple Hydrating Light Moisturizer, for Sensitive Skin, 4.2 Ounce, (Pack of 3)
16.0985 | Aveeno Ultra-Calming Daily BldGO Moisturizer For Sensitive Skin With Broad Spectrum Spf 15, 4 oz (Pack of 3)
15.4566 | Revitalizing Light-Weight Moisturizer SPF 15
14.9437 | Simple Hydrating Light Moisturizer, 4.2 Ounce 6-pack
14.5508 | BRTC Perfect Calming Cream 50ml, for Dry, Sensitive and Itchy Skin


In [41]:
results = search_bm25(bm25, products, "best shampoo for curly hair", top_k=5)
for r in results:
    print(f"{r['bm25_score']:.4f} | {r['title']}")

13.0654 | 2 pck of Hotheads Clean Shampoo 8 oz
12.2414 | Shea Moisture Jamaican Black Shampoo 13 Ounce (384ml) (Pack of 3)
12.2097 | J.R. Liggett's, Old Fashioned Bar, Shampoo, Jojoba & Peppermint, 3.5 oz (99 g) - 2pc
11.9237 | NORMADENSE 1 Prowash Thickening Shampoo. Normalizing Thickening Shampoo | Biotin Shampoo for, Dry, Weakened, Normal to Thin-Looking Hair | Vegan Hair Shampoo
11.8216 | Lee Stafford Bigger Fatter Fuller Volumizing Shampoo - For limp and fine hair


In [42]:
import sys
!{sys.executable} -m pip install sentence-transformers faiss-cpu numpy

In [ ]:
if 'src.semantic' in sys.modules:
    del sys.modules['src.semantic']

from src.semantic import build_semantic_index, search_semantic
from sentence_transformers import SentenceTransformer

# build index (will take 5-10 mins on CPU)
index, embeddings = build_semantic_index(corpus)

2026-05-03 13:05:56,014 Loading faiss.
2026-05-03 13:05:56,141 Successfully loaded faiss.

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/Users/komalpreet/miniconda3/envs/inflection/lib/python3.11/runpy.py", line 198, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/Users/komalpreet/miniconda3/envs/inflection/lib/python3.11/runpy.py", line 88, in _run_code
    exec(code, run_globals)
  File "/Users/komalpreet/miniconda3/envs/inflection/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_inst